In [1]:
# =========================
# 0) Install (Colab friendly)
# =========================
!pip -q install -U "transformers>=4.41" "datasets>=2.18" accelerate bitsandbytes peft PyPDF2 huggingface_hub
!pip -q install jsonlines tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 9.4 MB/s eta 0:00:00


In [2]:
# =========================
# 1) Imports + basic setup
# =========================
import os
import re
import warnings
import torch
import PyPDF2

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

warnings.filterwarnings("ignore")

# Reproducibility
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


In [3]:
# =========================
# 2) (Optional) Hugging Face auth
#    - 只在你要 push_to_hub 時需要
# =========================
from getpass import getpass
hfapi_key = getpass("Enter your HuggingFace access token (leave blank to skip): ").strip()

if hfapi_key:
    os.environ["HF_TOKEN"] = hfapi_key
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hfapi_key
    print("HF token set.")
else:
    print("HF token skipped (you can still train locally).")

Enter your HuggingFace access token (leave blank to skip): ··········
HF token set.


In [5]:
# =========================
# 3) Read PDF
#    - 修正點：extract_text() 可能回 None，需保護
#    - 保留你「從第5頁開始」的設定
# =========================
def read_pdf(pdf_path: str, start_page: int = 4) -> str:
    """
    start_page=4 => 0-based 第5頁開始
    """
    parts = []
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page_num in range(start_page, len(reader.pages)):
            page = reader.pages[page_num]
            page_text = page.extract_text() or ""
            parts.append(page_text)
    return "\n".join(parts)


pdf_path = "/InnoVait_TCM1.pdf"   # ← Colab 通常放 /content，請改成你的實際路徑
text_file = read_pdf(pdf_path, start_page=4)

print("Raw chars:", len(text_file))

Raw chars: 8626


In [6]:
# =========================
# 4) Clean text (保留你的 regex 清理)
# =========================
text_file = re.sub(r"\n+", "\n", text_file).strip()      # 多重換行
text_file = re.sub(r"[ \t]+", " ", text_file).strip()    # 多空白

# 移除頁眉頁碼（依你 PDF 格式調整）
text_file = re.sub(r" \d+ International Gita Society", "", text_file)
text_file = re.sub(r" Bhagavad -Gita \d+", "", text_file)

print("Clean chars:", len(text_file))

Clean chars: 8597


In [7]:
# =========================
# 5) 建立訓練樣本
#    - 修正點：你原本「每100詞換行」其實只是排版
#      在 Colab 上更建議：以段落聚合成樣本（較不切壞句子）
#    - 但如果你堅持要保留「100 words/line」也可以切回去
# =========================
def make_samples_by_paragraph(text: str, min_chars=200, max_chars=1500):
    paras = [p.strip() for p in text.split("\n") if p.strip()]
    samples, buf = [], ""
    for p in paras:
        if len(buf) + len(p) + 1 <= max_chars:
            buf = (buf + " " + p).strip()
        else:
            if len(buf) >= min_chars:
                samples.append(buf)
            buf = p
    if len(buf) >= min_chars:
        samples.append(buf)
    return samples

samples = make_samples_by_paragraph(text_file, min_chars=200, max_chars=1500)
print("Num samples:", len(samples))
print("Sample preview:\n", samples[0][:400])


Num samples: 6
Sample preview:
 5 InnovA i T Taking the pulse ( ) TCM practitioners place three fingers (index, middle and ring) along the radial artery of the patient to feel for the intensity, rate, rhythm, wave characteristic and resilience of the pulse (See Fig. 6). Up to 28 types of pulses have been described in the ancient archives of TCM like the Su Wen of the Yellow Emperor’s Internal Classic in 260 B C (Lu, 1985 ) and t


In [8]:
# =========================
# 6) Train/Val split
#    - 修正點：你原本用「字元長度」切 80/20 會切壞單字/句子
#      改成用 datasets 的 train_test_split
# =========================
from datasets import Dataset
ds = Dataset.from_dict({"text": samples}).train_test_split(test_size=0.2, seed=42)

# 存成 text 檔（保留你原本會輸出 train.txt / val.txt 的習慣）
train_file_path = "/content/train.txt"
val_file_path = "/content/val.txt"

with open(train_file_path, "w", encoding="utf-8") as f:
    f.write("\n".join(ds["train"]["text"]))
with open(val_file_path, "w", encoding="utf-8") as f:
    f.write("\n".join(ds["test"]["text"]))

print("Saved:", train_file_path, val_file_path)

Saved: /content/train.txt /content/val.txt


In [9]:
# =========================
# 7) Load dataset from text files (保留你原本 load_dataset("text") 的做法)
# =========================
dataset = load_dataset("text", data_files={"train": train_file_path, "validation": val_file_path})
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 4
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 2
    })
})


In [10]:
# =========================
# 8) Tokenizer
#    - 修正點：GPT2Tokenizer + pad_token=unk 不太理想
#      Colab 常見：用 AutoTokenizer，pad_token 設 eos_token
# =========================
checkpoint = "gpt2"  # 你原本是 openai-community/gpt2；gpt2 也可，Colab 更常用
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [11]:
# GPT-2 沒有 pad token，慣例用 eos 當 pad
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# =========================
# 9) Tokenize
#    - 修正點：你原本 padding='max_length' + return_tensors='pt' 在 dataset.map 可能不必要/容易怪
#      改成回傳 python list，交給 data collator 做 batch tensor
# =========================
block_size = 256

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=block_size,
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print("Tokenized columns:", tokenized_datasets["train"].column_names)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenized columns: ['input_ids', 'attention_mask']


In [12]:
# =========================
# 10) Data collator (causal LM)
# =========================
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)


# =========================
# 11) Model
#    - 修正點：用 AutoModelForCausalLM（取代 GPT2LMHeadModel / AutoModelWithLMHead）
# =========================
model = AutoModelForCausalLM.from_pretrained(checkpoint)

# 放到 GPU/CPU
model.to(device)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [13]:
# =========================
# 12) TrainingArguments (Colab 版)
#    - 修正點：num_train_epochs=100 太誇張，資料小會過擬合也跑超久
#      建議先 1~3 epochs 看效果
#    - 加上 eval/save/logging，Colab 更好控
# =========================
model_output_path = "/content/tutor_model"

training_args = TrainingArguments(
    output_dir=model_output_path,
    per_device_train_batch_size=8 if device=="cuda" else 2,
    per_device_eval_batch_size=8 if device=="cuda" else 2,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=5e-5,
    fp16=(device=="cuda"),
    eval_strategy="steps", # Changed from evaluation_strategy
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)

trainer.train()


# =========================
# 13) Save model + tokenizer
# =========================
saved_model_path = "/content/finetuned_aitutor_model"
trainer.save_model(saved_model_path)
tokenizer.save_pretrained(saved_model_path)
print("Saved to:", saved_model_path)


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/finetuned_aitutor_model


In [14]:
# =========================
# 14) Generate (修正點：用 max_new_tokens，避免 max_length 與 prompt 長度互相影響)
# =========================
@torch.no_grad()
def generate_response(model, tokenizer, prompt, max_new_tokens=120, temperature=0.7):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Load saved model back (保留你原本流程)
my_model_finetuned = AutoModelForCausalLM.from_pretrained(saved_model_path).to(device)
my_tokenizer_finetuned = AutoTokenizer.from_pretrained(saved_model_path)

prompt = "What are the four pillars of TCM diagnosis described in the document?\nAnswer"
print(generate_response(my_model_finetuned, my_tokenizer_finetuned, prompt))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

What are the four pillars of TCM diagnosis described in the document?
Answer: The most important pillar is that there is a clear understanding of what you have done, and why. It has been suggested that your actions may be contributing to mental illness or other problems such as depression or anxiety. But this does not mean that it will lead to any health benefits; rather, we must acknowledge that many factors can affect our ability even when these changes don't seem apparent at first glance. Here's an example from one study where people were given instructions on how to take their meds for symptoms (see Figure 1): "There was no difference between taking two pills every day after drinking


In [16]:
# =========================
# 15) (Optional) Push to Hugging Face Hub
# =========================
if hfapi_key:
    from huggingface_hub import login
    login(token=hfapi_key)

    # IMPORTANT: If you encounter a '403 Forbidden' error here, it usually means your Hugging Face token
    # does not have 'write' permissions. Please generate a new token with 'write' access from
    # https://huggingface.co/settings/tokens and update the 'hfapi_key' variable in cell `iPSduydUlQGu`.
    # Ensure 'my_repo' is either an existing repo you have write access to, or a new repo under your username.
    # For example: my_repo = "your_username/ai-tutor-towardsai-updated"
    my_repo = "fourze5798/ai-tutor-towardsai-updated"  # 你可改成 "username/repo"
    my_model_finetuned.push_to_hub(repo_id=my_repo, commit_message="Upload fine-tuned model (Colab version)")
    my_tokenizer_finetuned.push_to_hub(repo_id=my_repo, commit_message="Upload tokenizer (Colab version)")
    print("Pushed to Hub:", my_repo)

    # =========================
    # 16) Load back from Hub and test
    #    - 修正點：不要用 AutoModelWithLMHead（較舊）
    # =========================
    my_checkpoint = my_repo if "/" in my_repo else f"{os.environ.get('HF_USERNAME','')}/{my_repo}".strip("/")
    loaded_model = AutoModelForCausalLM.from_pretrained(my_repo).to(device)
    loaded_tokenizer = AutoTokenizer.from_pretrained(my_repo)

    prompt2 = "What are the four pillars of TCM diagnosis described in the document?\nAnswer"
    print(generate_response(loaded_model, loaded_tokenizer, prompt2))
else:
    print("Skip pushing/loading from Hub (no token).")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...zy0h7i8/model.safetensors:   3%|3         | 16.7MB /  498MB            

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

What are the four pillars of TCM diagnosis described in the document?
Answer: The first pillar is that you should be able to control your own body's response to stress. This includes getting used and using new medications, which can increase blood pressure levels. In addition, there are other factors such as age, physical activity, genetic predisposition and lifestyle choices (see here). It also has a psychological component when it comes down on people who suffer from depression or anxiety. There is no way to avoid having this disease because so much information about treatment depends on one's personal experience and how well we manage our bodies with medication. You might not have access to all these things if
